# 02-03 임베딩 레이어

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 번호를 벡터로 바꾸기

In [ ]:
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)  # 번호 10개, 벡터 길이 4
print(emb.weight.shape)  # (10, 4)

ids = torch.tensor([3, 7, 3])  # (3,)
vectors = emb(ids)  # (3, 4)
print(vectors)
print(torch.equal(vectors[0], emb.weight[3]))  # 3번 행을 그대로 꺼내요

## 2. 원-핫 벡터 × 선형 레이어와 같아요

In [ ]:
one_hot = F.one_hot(ids, num_classes=10).float()  # (3, 10)
print(one_hot)
via_matmul = one_hot @ emb.weight  # (3, 10) @ (10, 4) = (3, 4)
print(torch.allclose(via_matmul, vectors))

## 3. 배치와 시퀀스

In [ ]:
batch = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0]])  # (batch=2, seq_len=4)
out = emb(batch)  # (2, 4, 4) = (batch, seq_len, embedding_dim)
print(out.shape)

## 4. 임베딩도 학습돼요

In [ ]:
# 숫자 0~11을 3으로 나눈 나머지(0, 1, 2)로 분류하는 작은 문제
torch.manual_seed(0)
nums = torch.arange(12)  # (12,)
labels = nums % 3  # (12,)

class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(12, 2)  # 숫자마다 2차원 벡터
        self.out = nn.Linear(2, 3)

    def forward(self, ids):
        return self.out(self.emb(ids))  # (batch, 2) -> (batch, 3)

model = TinyClassifier()
before = model.emb.weight.detach().clone()  # (12, 2)
opt = torch.optim.Adam(model.parameters(), lr=0.05)
for step in range(300):
    loss = F.cross_entropy(model(nums), labels)
    opt.zero_grad()
    loss.backward()
    opt.step()
after = model.emb.weight.detach()  # (12, 2)
print(f"손실 {loss.item():.4f}")

In [ ]:
colors = ["tab:red", "tab:green", "tab:blue"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, w, title in [(axes[0], before, "before training"), (axes[1], after, "after training")]:
    for i in range(12):
        ax.scatter(w[i, 0], w[i, 1], color=colors[i % 3], s=200, edgecolor="k")
        ax.text(w[i, 0], w[i, 1], str(i), ha="center", va="center", color="white", weight="bold")
    ax.set_title(title + " (color = n % 3)")
plt.show()

In [ ]:
plt.figure(figsize=(3, 5))
plt.imshow(after, cmap="RdBu_r")
plt.colorbar()
plt.yticks(range(12))
plt.ylabel("id")
plt.xlabel("embedding dim (2)")
plt.title("emb.weight")
plt.show()